# TASK 1

Este script  servirá  para extraer metricas generales a traves de la API de MINKA. A traves del ID de un proyecto se extraeran los siguientes parámetros.


* Número de observaciones total
* Número de personas que han subido observaciones
* Número de especies registradas
* Número de personas que han contribuido con identificaciones
* Top 3 de personas con más observaciones
* Últimas 5 especies diferentes registradas (especies de las últimas observaciones, si se repite la especie, buscas la siguiente hasta sacar las últimas 5 diferentes)
* Número de observaciones mensuales desde enero 2024 (por fecha de observación, no fecha de subida, fíjate en los filtros de fecha), en formato tabla (dataframe de pandas):


In [ ]:
import requests
import pandas as pd
import math 

In [ ]:
API_PATH = 'https://api.minka-sdg.org/v1/'

NÚMERO TOTAL DE OBSERVACIONES

In [ ]:
def get_project_observations(project_id : int):
    '''Esta función recibe el id de un pryecto y devuelve el número total de observaciones'''
    url_project_id = f'{API_PATH}observations?project_id={project_id}'
    try:
        obs_count = requests.get(url_project_id).json()['total_results']
        return obs_count
    except KeyError:
        return 0

In [ ]:
get_project_observations(283)

OBSERVACIONES DE UN RANGO DE PROYECTOS + EL NÚMERO TOTAL DE OBSERVACIONES

In [ ]:
total_obs = 0
for i in range(250, 285):
    obs_count = get_project_observations(i)
    total_obs += obs_count
    print(f'El número total de observaciones del proyecto {i} son: {obs_count}')
print(f"Total de observaciones: {total_obs}")

In [ ]:
def get_proj_json(project_id):
    url_id = f'{API_PATH}/observations?project_id={project_id}'
    requests.get(url_id).json()
    json_project = requests.get(url_id).json()['latitude']
    return json_project

* NUMERO DE PERSONAS QUE HAN SUBIDO OBSERVACIONES

In [ ]:
def get_project_observers(project_id: int):

    '''Esta función recibe el id de un proyecto y devuelve el número total de observadores'''

    url_observers = f'{API_PATH}observations/observers?project_id={project_id}'
    total_results = requests.get(url_observers).json()['total_results']
    return print(f'El número total de observadores del proyecto https://minka-sdg.org/projects/{project_id} son: {total_results}')

In [ ]:
get_project_observers(264)

* NÚMERO DE ESPECIES REGISTRADAS

In [ ]:
def get_project_species(project_id: int):

    '''Esta función recibe el id de un proyecto y devuelve el número total de especies'''

    url_species = f'{API_PATH}observations/species_counts?project_id={project_id}'   
    response = requests.get(url_species).json()['total_results']
    return print(f'El número total de especies registradas en el proyecto https://minka-sdg.org/projects/{project_id} son : {response}')

In [ ]:
get_project_species(264)

* NÚMERO DE PERSONAS QUE HAN CONTRIBUIDO CON IDENTIFICACIONES

In [ ]:
def get_project_identifiers(project_id: int):
    
    '''Esta función recibe el id de un proyecto y devuelve el número total de identificadores'''

    url_identifiers = f'{API_PATH}observations/identifiers?project_id={project_id}'
    response = requests.get(url_identifiers).json()['total_results']
    return print(f'El número total de identificadores del proyecto https://minka-sdg.org/projects/{project_id} son : {response}')

get_project_identifiers(264)

* TOP 3 PERSONAS CON MÁS OBSERVACIONES

In [31]:
def get_top_observers(project_id, num_top_users: int):
    
    url_observers = f'{API_PATH}observations/observers?project_id={project_id}'
    results = requests.get(url_observers).json()['results']
    print(f'TOP {num_top_users} PERSONAS CON MÁS OBSERVACIONES')
    top_users = []
    for i in range(num_top_users):
        user_data = results[i]['user']
        username = user_data['login']
        user_observations = results[i]['observation_count']
        top_users.append([username, user_observations])
        
    df = pd.DataFrame(top_users, columns = ['Usuario', 'Número de observaciones'])
    return df

In [32]:
get_top_observers(264, 5)

TOP 5 PERSONAS CON MÁS OBSERVACIONES


,Usuario,Número de observaciones
0,mediambient_ajelprat,1725
1,amb_platges,727
2,romu_freediving_photography,553
3,ngoncubells,326
4,xasalva,283


In [ ]:
import matplotlib.pyplot as plt

df_top_users = pd.DataFrame(top_users)
df_top_users = df_top_users.set_index('username')
df_top_users.plot(kind='bar', title='TOP 3 OBSERVADORES', legend=False)
plt.show()

* ULTIMAS 5 ESPECIES REGISTRADAS

In [ ]:
def get_last_species(project_id : int, num_species: int):

        ''''Esta función recibe el id de un proyecto y devuelve las últimas especies registradas
        Independientemente de que sean nuevas. Solo se anotarán las que tenga el quality_grade en 'research' ''' 
        dataframe = []
        url_species_reg = f'{API_PATH}observations?project_id={project_id}'
        results = requests.get(url_species_reg).json()['results']
        for i in range(num_species):
            if results[i]['quality_grade'] == 'research':
                species_name = results[i]['taxon']['name']
                species_obs_time = results[i]['observed_on']
                obs_made_by = results[i]['user']['login']
                #obs_image = results[i]['photos'][i]['url']
                #print(f'Especie: {i+1} {species_name}Fecha de la Observación: {species_obs_time}, Observada por: {obs_made_by}')

                dataframe.append([species_name, species_obs_time, obs_made_by])
            else:
                 continue
        
        df = pd.DataFrame(dataframe,columns = ['Especie', 'Fecha de observación', 'Observado por' ])
        return df


In [ ]:
get_last_species(264,30)

*  Número de observaciones mensuales desde enero 2024

In [ ]:
OBSERVATIONS_PATH = 'https://api.minka-sdg.org/v1/observations'

year = input('Inserta el año de observación que deseas consultar :')

url_observations_month = f'{OBSERVATIONS_PATH}/histogram?project_id={264}&year={year}&date_field=observed&interval=month_of_year'

response = requests.get(url_observations_month).json()

data = response['results']

df = pd.DataFrame(
    {"mes": [f"{year}-{int(m):02d}" for m in data["month_of_year"].keys()],
     "num_observations": list(data["month_of_year"].values())}
)

print(df)

### MÉTRICAS ADICIONALES

* Top 5 espécies más observadas
* Top 3 localizaciones con más observaciones
* Espécies observadas en peligro de extinción/amenazadas
* Cantidad de nuevos usuarios que han hecho observaciones en el ultimo mes/semana
* Observaciones nuevas en el útlimo mes / semana

* TOP 5 ESPÉCIES MÁS OBSERVADAS

In [39]:
# Para obtener el top 5 espécies más observadas y el número total de observaciones realizamos un procedimiento similar al del TOP 3 personas con más obseravciones y el código de espécies registradas.

def get_top_observations(project_id: int, num_top_observations: int):
    '''Esta función recibe el id del proyecto junto al numero de especies observadas que se quieren visualizar
    y devuelve el top con las especies más observadas que tengan quality_grade en 'research' ''' 
    top_species = []
    url_top_species = f'{API_PATH}observations/species_counts?project_id={project_id}'
    results = requests.get(url_top_species).json()['results']

    for i in range(num_top_observations):
        if results[i]['taxon']['rank_level'] == 10 :
            species_name = results[i]['taxon']['name']
            species_count = results[i]['count']
            top_species.append([species_name, species_count])
        else:
            continue

    df = pd.DataFrame(top_species, columns = ['Especie', 'Num de observaciones' ])
    return df
        

In [40]:
get_top_observations(264, 5)

,Especie,Num de observaciones
0,Acanthocardia tuberculata,107
1,Spisula subtruncata,104
2,Velella velella,92
3,Stramonita haemastoma,90
4,Paracentrotus lividus,81


In [ ]:
requests.get(url_observations, params={'page': 1, 'per_page':200}).json()['results'][0]['id']

In [ ]:
# Función tome project_id como parametro
# Devuelva dataframe de observaciones con grado investigación
# id_obs, user_id, observed_on, created_at, taxon_id, taxon_name, latitude, longitude

In [ ]:
per_page = 200
total_number = requests.get(url_observations).json()['total_results']
for i in range(total_number / 200 + 2):
    results = requests.get(url_observations, params={'page': i, 'per_page':200}).json()['results']
    for result in results:
        result['id']


In [ ]:
response = requests.get(url_observations, params={'page': page, 'per_page':30}).json()
response

In [ ]:
all_places_ids

In [ ]:
# Documentación: https://geopy.readthedocs.io/en/stable/

from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="minka-project")

location = geolocator.reverse("52.509669, 13.376294")

print(location.address)